# HaNoRec x LLM2Rec CF-hardness real technical smoke run

Real SASRec retriever (frozen), real Qwen2.5-VL-3B SFT+DPO with ported HaRS/CF hardness and NoDO, on real Games_5core data. Technical smoke test per plans/260918-1114-hanorec-cf-hardness/plan.md -- not a research-effectiveness claim.


In [ ]:
from __future__ import annotations
import subprocess, sys, time
DEADLINE = time.monotonic() + 41000.0
PINS = ["transformers==4.51.3", "peft==0.15.2", "accelerate==1.6.0", "bitsandbytes==0.45.5"]
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *PINS], check=True)
import torch
print({"cuda": torch.cuda.is_available(), "device_count": torch.cuda.device_count(), "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
if not torch.cuda.is_available():
    raise RuntimeError("This kernel requires exactly one visible GPU; none detected.")


## prep.py (verbatim, real data preparation)


In [ ]:
"""Real data preparation for the HaNoRec x LLM2Rec CF-hardness smoke pipeline.

Every artifact loaded here is the genuine pinned LLM2Rec Games_5core release
(commit 73b481f710f67166ab958f4985d27b27fb410871): the frozen SASRec checkpoint
(seed 2024, IEM checkpoint-500 title embeddings), the real train/val/test_data.txt
sequences, item_titles.json, and the real Amazon image manifest produced by the
G1 visual preflight screen. No synthetic catalog, no invented item ids.

The SASRec reconstruction below is a verbatim functional port of
seqrec/base.py, seqrec/modules.py (TransformerEncoder_v2 subset),
seqrec/models/Embedding2.py, and seqrec/models/SASRec/_model.py from that same
pinned commit -- copied because Kaggle kernel_sources mount only output files,
never source code (rule://kaggle-mcp-experiments), so this kernel must stay
self-contained. Parameter names/shapes are kept identical to the original so
the pinned .pth state_dict loads without remapping.
"""
from __future__ import annotations

import copy
import hashlib
import json
import math
import urllib.request
from pathlib import Path
from typing import Any


# --------------------------------------------------------------------------
# Verbatim-ported SASRec architecture (see module docstring for provenance).
# --------------------------------------------------------------------------

def _build_sasrec_model(config: dict[str, Any], pretrained_embeddings):
    import torch
    import torch.nn as nn

    class FeedForward(nn.Module):
        def __init__(self, hidden_size, inner_size, hidden_dropout_prob, layer_norm_eps=1e-12):
            super().__init__()
            self.dense_1 = nn.Linear(hidden_size, inner_size)
            self.dense_2 = nn.Linear(inner_size, hidden_size)
            self.LayerNorm = nn.LayerNorm(hidden_size, eps=layer_norm_eps)
            self.dropout = nn.Dropout(hidden_dropout_prob)

        def forward(self, input_tensor):
            hidden_states = self.dense_1(input_tensor)
            hidden_states = hidden_states * 0.5 * (1.0 + torch.erf(hidden_states / math.sqrt(2.0)))
            hidden_states = self.dense_2(hidden_states)
            hidden_states = self.dropout(hidden_states)
            return self.LayerNorm(hidden_states + input_tensor)

    class MultiHeadAttention_v2(nn.Module):
        def __init__(self, n_heads, hidden_size, hidden_dropout_prob, attn_dropout_prob, layer_norm_eps=1e-12):
            super().__init__()
            if hidden_size % n_heads != 0:
                raise ValueError("hidden_size must be divisible by n_heads")
            self.num_attention_heads = n_heads
            self.attention_head_size = hidden_size // n_heads
            self.all_head_size = self.num_attention_heads * self.attention_head_size
            self.sqrt_attention_head_size = math.sqrt(self.attention_head_size)
            self.query = nn.Linear(hidden_size, self.all_head_size)
            self.key = nn.Linear(hidden_size, self.all_head_size)
            self.value = nn.Linear(hidden_size, self.all_head_size)
            self.softmax = nn.Softmax(dim=-1)
            self.attn_dropout = nn.Dropout(attn_dropout_prob)
            self.dense = nn.Linear(hidden_size, hidden_size)
            self.LayerNorm = nn.LayerNorm(hidden_size, eps=layer_norm_eps)
            self.out_dropout = nn.Dropout(hidden_dropout_prob)

        def _transpose(self, x):
            shape = x.size()[:-1] + (self.num_attention_heads, self.attention_head_size)
            return x.view(*shape)

        def forward(self, input_tensor, attention_mask):
            query_layer = self._transpose(self.query(input_tensor)).permute(0, 2, 1, 3)
            key_layer = self._transpose(self.key(input_tensor)).permute(0, 2, 3, 1)
            value_layer = self._transpose(self.value(input_tensor)).permute(0, 2, 1, 3)
            attention_scores = torch.matmul(query_layer, key_layer) / self.sqrt_attention_head_size
            attention_scores = attention_scores + attention_mask
            attention_probs = self.attn_dropout(self.softmax(attention_scores))
            context_layer = torch.matmul(attention_probs, value_layer).permute(0, 2, 1, 3).contiguous()
            new_shape = context_layer.size()[:-2] + (self.all_head_size,)
            context_layer = context_layer.view(*new_shape)
            hidden_states = self.out_dropout(self.dense(context_layer))
            return self.LayerNorm(hidden_states + input_tensor)

    class TransformerLayer_v2(nn.Module):
        def __init__(self, n_heads, hidden_size, intermediate_size, hidden_dropout_prob, attn_dropout_prob):
            super().__init__()
            self.multi_head_attention = MultiHeadAttention_v2(
                n_heads, hidden_size, hidden_dropout_prob, attn_dropout_prob
            )
            self.feed_forward = FeedForward(hidden_size, intermediate_size, hidden_dropout_prob)

        def forward(self, hidden_states, attention_mask):
            attention_output = self.multi_head_attention(hidden_states, attention_mask)
            return self.feed_forward(attention_output)

    class TransformerEncoder_v2(nn.Module):
        def __init__(self, cfg):
            super().__init__()
            layer = TransformerLayer_v2(cfg["num_heads"], cfg["hidden_size"], 256, cfg["dropout"], cfg["dropout"])
            self.layer = nn.ModuleList([copy.deepcopy(layer) for _ in range(cfg["layer_num"])])

        def forward(self, hidden_states, attention_mask):
            layers = []
            for layer_module in self.layer:
                hidden_states = layer_module(hidden_states, attention_mask)
                layers.append(hidden_states)
            return layers

    def get_attention_mask(item_seq):
        attention_mask = item_seq != 0
        extended = attention_mask.unsqueeze(1).unsqueeze(2)
        extended = torch.tril(extended.expand((-1, -1, item_seq.size(-1), -1)))
        return torch.where(extended, torch.tensor(0.0), torch.tensor(-10000.0))

    def gather_indexes(output, gather_index):
        gather_index = gather_index.view(-1, 1, 1).expand(-1, -1, output.shape[-1])
        return output.gather(dim=1, index=gather_index).squeeze(1)

    class Embedding2Weight:
        __slots__ = ("data",)

        def __init__(self, data):
            self.data = data

    class Embedding2(nn.Module):
        def __init__(self, adapter, embedding):
            super().__init__()
            self.embedding = embedding
            self.adapter = adapter

        def forward(self, indices):
            return self.adapter(self.embedding(indices))

        @property
        def weight(self):
            return Embedding2Weight(self.adapter(self.embedding.weight.data))

    class SASRec(nn.Module):
        def __init__(self, cfg, pretrained_embs):
            super().__init__()
            self.config = cfg
            assert pretrained_embs.shape[0] == cfg["item_num"] + 1
            self.pretrained_item_embeddings = nn.Embedding.from_pretrained(pretrained_embs, padding_idx=0)
            self.pretrained_item_embeddings.weight.requires_grad = False
            assert cfg["adapter_dims"][-1] == -1
            mlp_dims = [pretrained_embs.shape[-1]] + list(cfg["adapter_dims"])
            mlp_dims[-1] = cfg["hidden_size"]
            self.item_embeddings_adapter = nn.Sequential()
            self.item_embeddings_adapter.add_module("linear_0", nn.Linear(mlp_dims[0], mlp_dims[1]))
            for index in range(1, len(mlp_dims) - 1):
                self.item_embeddings_adapter.add_module(f"activation_{index}", nn.ReLU())
                self.item_embeddings_adapter.add_module(f"linear_{index}", nn.Linear(mlp_dims[index], mlp_dims[index + 1]))
            self.item_embeddings = Embedding2(self.item_embeddings_adapter, self.pretrained_item_embeddings)
            self.positional_embeddings = nn.Embedding(cfg["max_seq_length"], cfg["hidden_size"])
            self.emb_dropout = nn.Dropout(cfg["dropout"])
            self.transformer_encoder = TransformerEncoder_v2(cfg)

        def get_embeddings(self, items):
            return self.item_embeddings(items)

        def get_all_embeddings(self):
            return self.item_embeddings.weight.data

        def get_representation(self, item_seqs, seq_lengths):
            inputs_emb = self.get_embeddings(item_seqs)
            inputs_emb = inputs_emb + self.positional_embeddings(torch.arange(self.config["max_seq_length"]))
            seq = self.emb_dropout(inputs_emb)
            mask = get_attention_mask(item_seqs)
            layers = self.transformer_encoder(seq, mask)
            output = layers[-1]
            return gather_indexes(output, seq_lengths - 1)

        def score_full_catalog(self, item_seqs, seq_lengths):
            state_hidden = self.get_representation(item_seqs, seq_lengths)
            test_item_emb = self.get_all_embeddings()
            select_pool = self.config["select_pool"]
            scores = torch.matmul(state_hidden, test_item_emb.transpose(0, 1))
            return scores[:, select_pool[0]:select_pool[1]]

    return SASRec(config, pretrained_embeddings)


def _read_sequences(path: Path) -> list[list[int]]:
    lines = path.read_text(encoding="utf-8").splitlines()
    return [list(map(int, line.split())) for line in lines if line.strip()]


def _sha256_bytes(payload: bytes) -> str:
    return hashlib.sha256(payload).hexdigest()


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _find_one(root: Path, name: str) -> Path:
    matches = sorted(path for path in root.rglob(name) if path.is_file())
    if not matches:
        raise FileNotFoundError(f"required real artifact not found under {root}: {name}")
    return matches[0]


def _verify_pin(path: Path, pins: dict[str, str]) -> None:
    expected = pins.get(path.name)
    if expected is None:
        return
    actual = _sha256_file(path)
    if actual != expected:
        raise RuntimeError(f"artifact hash mismatch for {path}: expected {expected}, got {actual}")


def _future_items_for_prefix(history_and_target: list[int], all_rows: list[list[int]]) -> set[int]:
    """Real false-negative mitigation: items the same underlying user interacts
    with later, detected by rows that extend this exact row as a strict prefix
    (see plans/260918-1114-hanorec-cf-hardness/phase-01-local-spec-and-audit.md#3).
    """
    future: set[int] = set()
    prefix_len = len(history_and_target)
    for row in all_rows:
        if len(row) > prefix_len and row[:prefix_len] == history_and_target:
            future.update(row[prefix_len:])
    return future


def _download_image(url: str, cache_dir: Path) -> tuple[str, str]:
    cache_dir.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(url.encode("utf-8")).hexdigest()[:16]
    suffix = Path(url).suffix or ".jpg"
    target = cache_dir / f"{digest}{suffix}"
    if not target.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "hanorec-cf-hardness-prep/1.0"})
        with urllib.request.urlopen(request, timeout=30) as response:
            payload = response.read()
        if not payload:
            raise RuntimeError(f"empty image payload for {url}")
        target.write_bytes(payload)
    return str(target), _sha256_file(target)


def prepare(config: dict[str, Any], input_root: Path, output_root: Path, source_root: Path) -> dict[str, Any]:
    """Build the real `prepared` dict per the frozen contract.

    input_root: directory that contains (recursively) the real downloaded
    LLM2Rec Games_5core artifacts -- games_evaluation_artifact.json, the
    real train/val/test_data.txt + item_titles.json, the pinned SASRec
    .pth checkpoint, the pinned title embedding .npy, and the real
    Video_Games_image_manifest.jsonl. source_root is unused here (SASRec is
    ported inline); kept for contract symmetry with train.py.
    """
    import numpy as np
    import torch

    output_root.mkdir(parents=True, exist_ok=True)
    pins: dict[str, str] = dict(config.get("artifact_pins", {}))

    checkpoint_path = _find_one(input_root, config["checkpoint_name"])
    embedding_path = _find_one(input_root, config["embedding_name"])
    manifest_path = _find_one(input_root, "Video_Games_image_manifest.jsonl")
    train_path = _find_one(input_root, "train_data.txt")
    val_path = _find_one(input_root, "val_data.txt")
    test_path = _find_one(input_root, "test_data.txt")
    full_data_path = _find_one(input_root, "data.txt")
    titles_path = _find_one(input_root, "item_titles.json")

    for path in (checkpoint_path, embedding_path, manifest_path, train_path, val_path, test_path, titles_path):
        _verify_pin(path, pins)

    item_titles: dict[str, str] = json.loads(titles_path.read_text(encoding="utf-8"))

    max_seq_length = int(config["max_seq_length"]) if "max_seq_length" in config else 10
    full_rows = _read_sequences(full_data_path)
    total_item_num = max(item for row in full_rows for item in row)

    train_rows_full = _read_sequences(train_path)
    val_rows_full = _read_sequences(val_path)
    test_rows_full = _read_sequences(test_path)
    train_rows = [row[-(max_seq_length + 1):] for row in train_rows_full]
    val_rows = [row[-(max_seq_length + 1):] for row in val_rows_full]
    test_rows = [row[-(max_seq_length + 1):] for row in test_rows_full]

    embedding_matrix = np.load(embedding_path)
    if embedding_matrix.shape[0] != total_item_num + 1:
        raise RuntimeError(
            f"embedding row count {embedding_matrix.shape[0]} does not match data.txt-derived "
            f"item_num+1 {total_item_num + 1}"
        )
    pretrained_embeddings = torch.tensor(embedding_matrix, dtype=torch.float32)

    sasrec_config = {
        "hidden_size": 128,
        "layer_num": 2,
        "num_heads": 2,
        "dropout": 0.3,
        "adapter_dims": [-1],
        "max_seq_length": max_seq_length,
        "item_num": total_item_num,
        "select_pool": [1, total_item_num + 1],
    }
    model = _build_sasrec_model(sasrec_config, pretrained_embeddings)
    state_dict = torch.load(checkpoint_path, map_location="cpu")
    model.load_state_dict(state_dict, strict=True)
    model.eval()

    def score_rows(rows: list[list[int]]) -> "np.ndarray":
        item_seqs = []
        seq_lengths = []
        for row in rows:
            history = row[:-1]
            seq_length = len(history)
            padded = history + [0] * (max_seq_length - seq_length)
            item_seqs.append(padded)
            seq_lengths.append(seq_length)
        item_seqs_tensor = torch.tensor(item_seqs, dtype=torch.long)
        seq_lengths_tensor = torch.tensor(seq_lengths, dtype=torch.long)
        with torch.no_grad():
            scores = model.score_full_catalog(item_seqs_tensor, seq_lengths_tensor)
        return scores.cpu().numpy()

    seed = int(config.get("seed", 2024))
    history_items = int(config.get("history_items", 3))
    train_pair_target = int(config.get("train_pairs", 8))
    eval_user_target = int(config.get("eval_users", 4))
    top_m = int(config.get("top_m", 20))

    train_candidates = [
        (index, row) for index, row in enumerate(train_rows) if len(row) - 1 >= history_items
    ]
    if len(train_candidates) < train_pair_target:
        raise RuntimeError(
            f"only {len(train_candidates)} real train rows qualify with history_items>={history_items}; "
            f"need {train_pair_target}"
        )
    selected_train = train_candidates[:train_pair_target]
    selected_train_full_rows = [train_rows_full[index] for index, _ in selected_train]
    selected_train_scores = score_rows([row for _, row in selected_train])

    train_pairs: list[dict[str, Any]] = []
    for position, (row_index, row) in enumerate(selected_train):
        history_full = row[:-1]
        target = row[-1]
        history = history_full[-history_items:]
        future_excluded = _future_items_for_prefix(
            selected_train_full_rows[position][: len(selected_train_full_rows[position]) - 1] + [target],
            train_rows_full,
        )
        scores = selected_train_scores[position]
        excluded_ids = {target} | future_excluded | {0}
        ranked = np.argsort(-scores)
        negative_item = None
        for rank_index in ranked:
            candidate_item = int(rank_index) + 1
            if candidate_item not in excluded_ids:
                negative_item = candidate_item
                break
        if negative_item is None:
            raise RuntimeError(f"could not find a real negative for train row {row_index}")
        cf_margin = float(scores[target - 1] - scores[negative_item - 1])
        train_pairs.append(
            {
                "user_id": f"train_row_{row_index}",
                "history": [int(item) for item in history],
                "positive": int(target),
                "negative": int(negative_item),
                "cf_margin": cf_margin,
            }
        )

    eval_candidates = [
        (index, row) for index, row in enumerate(test_rows) if len(row) - 1 >= history_items
    ]
    if len(eval_candidates) < eval_user_target:
        raise RuntimeError(
            f"only {len(eval_candidates)} real test rows qualify with history_items>={history_items}; "
            f"need {eval_user_target}"
        )
    selected_eval = eval_candidates[:eval_user_target]
    selected_eval_scores = score_rows([row for _, row in selected_eval])

    evaluation_rows: list[dict[str, Any]] = []
    for position, (row_index, row) in enumerate(selected_eval):
        history_full = row[:-1]
        target = row[-1]
        history = history_full[-history_items:]
        scores = selected_eval_scores[position]
        top_indices = np.argsort(-scores)[:top_m]
        candidates = [int(index) + 1 for index in top_indices]
        candidate_scores = [float(scores[index]) for index in top_indices]
        evaluation_rows.append(
            {
                "user_id": f"test_row_{row_index}",
                "history": [int(item) for item in history],
                "target": int(target),
                "candidates": candidates,
                "retriever_scores": candidate_scores,
            }
        )

    referenced_items: set[int] = set()
    for pair in train_pairs:
        referenced_items.update(pair["history"])
        referenced_items.add(pair["positive"])
        referenced_items.add(pair["negative"])
    for row in evaluation_rows:
        referenced_items.update(row["history"])
        referenced_items.add(row["target"])
        referenced_items.update(row["candidates"])

    manifest_by_item: dict[int, dict[str, Any]] = {}
    with manifest_path.open("r", encoding="utf-8") as handle:
        for line in handle:
            record = json.loads(line)
            manifest_by_item[int(record["item_id"])] = record

    image_cache_dir = output_root / "images"
    catalog: dict[str, dict[str, str]] = {}
    image_provenance: dict[str, dict[str, str]] = {}
    for item_id in sorted(referenced_items):
        title = item_titles.get(str(item_id))
        if title is None:
            raise RuntimeError(f"real item_titles.json has no entry for referenced item {item_id}")
        record = manifest_by_item.get(item_id)
        if record is None or record.get("status") != "ok" or not record.get("url"):
            raise RuntimeError(f"real image manifest has no usable entry for referenced item {item_id}")
        local_path, image_sha256 = _download_image(record["url"], image_cache_dir)
        catalog[str(item_id)] = {"title": title, "image_path": local_path}
        image_provenance[str(item_id)] = {
            "url": record["url"],
            "parent_asin": record.get("parent_asin", ""),
            "sha256": image_sha256,
        }

    provenance = {
        "seed": seed,
        "checkpoint": {"path": str(checkpoint_path), "sha256": _sha256_file(checkpoint_path)},
        "embedding": {"path": str(embedding_path), "sha256": _sha256_file(embedding_path)},
        "train_data": {"path": str(train_path), "sha256": _sha256_file(train_path)},
        "val_data": {"path": str(val_path), "sha256": _sha256_file(val_path)},
        "test_data": {"path": str(test_path), "sha256": _sha256_file(test_path)},
        "item_titles": {"path": str(titles_path), "sha256": _sha256_file(titles_path)},
        "manifest": {"path": str(manifest_path), "sha256": _sha256_file(manifest_path)},
        "total_item_num": total_item_num,
        "sasrec_config": sasrec_config,
        "train_pair_construction": (
            "positive/negative drawn from real train_data.txt rows; negative is the frozen "
            "SASRec model's own highest-scoring non-target, non-future item over the full "
            "real catalog; future items detected via exact-prefix row extension in "
            "train_data.txt (false-negative mitigation, phase-01 section 3)"
        ),
        "eval_construction": (
            "real test_data.txt rows; candidates are the frozen SASRec model's real top-M "
            "full-catalog ranking; target never inserted artificially"
        ),
        "history_items": history_items,
        "images_downloaded": len(image_provenance),
        "image_provenance": image_provenance,
    }

    return {
        "catalog": catalog,
        "train": train_pairs,
        "evaluation": evaluation_rows,
        "provenance": provenance,
    }


## train.py (verbatim, real SFT+DPO training and reranking)


In [ ]:
"""Real Qwen2.5-VL SFT+DPO training with HaRS/CF hardness and NoDO noise.

Design decision (yes/no binary framing, matching upstream HaNoRec `--hit 1`):
for every real (history, positive, negative) pair from prep.py, two DPO
examples are built that SHARE the exact same prompt template (history titles
+ images, one candidate image/title, one yes/no question) and differ only in
which candidate is shown and which answer is therefore "chosen":

  - candidate = positive item -> chosen="Yes", rejected="No"
  - candidate = negative item -> chosen="No",  rejected="Yes"

This keeps chosen/rejected sharing identical prompt structure per pair while
reusing the paper's own binary next-item preference task, and it makes
reranking simple and real: for every eval candidate, run one forward pass of
the SAME prompt template and rank by score = logit("Yes") - logit("No").

HaRS semantic hardness and NoDO perturbation are ported verbatim (pure math
functions copied, not imported -- Kaggle kernel_sources mount output files
only, never source code, rule://kaggle-mcp-experiments) from the pinned
HaNoRec commit's hanorec/hars/math.py, hanorec/hars/hardness.py, and
hanorec/nodo/hooks.py.
"""
from __future__ import annotations

import json
import math
import time
from pathlib import Path
from typing import Any


# --------------------------------------------------------------------------
# Verbatim-ported HaRS math (hanorec/hars/math.py, commit 587face7).
# --------------------------------------------------------------------------

def _finite(values, name):
    checked = [float(v) for v in values]
    if not checked:
        raise ValueError(f"{name} must be non-empty")
    if not all(math.isfinite(v) for v in checked):
        raise ValueError(f"{name} must contain only finite values")
    return checked


def _stable_sigmoid(value: float) -> float:
    if value >= 0:
        return 1.0 / (1.0 + math.exp(-value))
    exp_value = math.exp(value)
    return exp_value / (1.0 + exp_value)


def hars_softmax(values):
    checked = _finite(values, "values")
    offset = max(checked)
    weights = [math.exp(v - offset) for v in checked]
    total = sum(weights)
    return [w / total for w in weights]


def hars_probability_distance(chosen, rejected):
    chosen_v = _finite(chosen, "chosen")
    rejected_v = _finite(rejected, "rejected")
    if len(chosen_v) != len(rejected_v):
        raise ValueError("chosen and rejected must have equal length")
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(chosen_v, rejected_v)))


def hars_normalize_hardness(deltas):
    checked = _finite(deltas, "deltas")
    if any(d < 0 for d in checked):
        raise ValueError("deltas must be non-negative")
    denominator = _stable_sigmoid(sum(checked) / len(checked))
    return [_stable_sigmoid(d) / denominator for d in checked]


def cf_normalize_hardness(margins):
    """Frozen Phase 1 formula: lambda_cf = sigmoid(m) / sigmoid(mean(m)).

    Unlike hars_normalize_hardness, margins are signed (a retriever score
    difference), so this is intentionally not clamped to (0, 1); see
    plans/260918-1114-hanorec-cf-hardness/phase-01-local-spec-and-audit.md#2.
    """
    checked = _finite(margins, "margins")
    denominator = _stable_sigmoid(sum(checked) / len(checked))
    return [_stable_sigmoid(m) / denominator for m in checked]


def responsiveness(reward_gaps):
    """Torch-free port of hanorec/hars/math.py::model_responsiveness (Eq. 8)."""
    checked = _finite(reward_gaps, "reward_gaps")
    eps = 1e-8
    mean_gap = sum(checked) / len(checked)
    if abs(mean_gap) > eps:
        scale = mean_gap
    else:
        scale = max(sum(abs(v) for v in checked) / len(checked), eps)
    normalized = [v / scale for v in checked]
    trimmed = sorted(normalized)[1:-1] if len(normalized) > 2 else normalized
    trimmed_mean = sum(trimmed) / len(trimmed)
    normalized_mean = sum(normalized) / len(normalized)
    return _stable_sigmoid(trimmed_mean) / _stable_sigmoid(normalized_mean)


# --------------------------------------------------------------------------
# Prompt construction shared by SFT, DPO, and reranking.
# --------------------------------------------------------------------------

_QUESTION = "Based on the user's history, will they like this candidate item next? Answer Yes or No."


def _resize_image(image_path: str, max_pixels: int):
    from PIL import Image

    with Image.open(image_path) as handle:
        image = handle.convert("RGB")
    width, height = image.size
    if width * height > max_pixels:
        scale = (max_pixels / (width * height)) ** 0.5
        image = image.resize((max(1, int(width * scale)), max(1, int(height * scale))))
    return image


def _build_messages(catalog: dict[str, dict[str, str]], history: list[int], candidate: int, max_pixels: int, shuffle_map: dict[int, int] | None):
    def item_image(item_id: int):
        source_id = item_id if shuffle_map is None else shuffle_map.get(item_id, item_id)
        return _resize_image(catalog[str(source_id)]["image_path"], max_pixels)

    content: list[dict[str, Any]] = [{"type": "text", "text": "User history:"}]
    for history_item in history:
        content.append({"type": "image", "image": item_image(history_item)})
        content.append({"type": "text", "text": catalog[str(history_item)]["title"]})
    content.append({"type": "text", "text": "Candidate item:"})
    content.append({"type": "image", "image": item_image(candidate)})
    content.append({"type": "text", "text": catalog[str(candidate)]["title"]})
    content.append({"type": "text", "text": _QUESTION})
    return [{"role": "user", "content": content}]


def _semantic_embedding(model, processor, torch_module, title: str, image):
    tokenizer = processor.tokenizer
    inputs = tokenizer(title, return_tensors="pt", truncation=True)
    device = next(model.parameters()).device
    with torch_module.no_grad():
        text_states = model.model.embed_tokens(inputs["input_ids"].to(device))
        mask = inputs["attention_mask"].to(device).unsqueeze(-1).to(text_states.dtype)
        text_vector = (text_states * mask).sum(dim=1) / mask.sum(dim=1).clamp_min(1)

        image_inputs = processor.image_processor(images=[image], return_tensors="pt")
        pixel_values = image_inputs["pixel_values"].to(device)
        grid_thw = image_inputs["image_grid_thw"].to(device)
        visual_dtype = next(model.visual.parameters()).dtype
        pixel_values = pixel_values.to(visual_dtype)
        visual_features = model.visual(pixel_values, grid_thw)
        visual_vector = visual_features.mean(dim=0, keepdim=True).to(text_vector.dtype)

    text_np = text_vector[0].float().cpu().numpy()
    visual_np = visual_vector[0].float().cpu().numpy()
    return text_np, visual_np


def _fuse_and_topk(text_matrix, visual_matrix, item_order: list[int], k: int):
    import numpy as np

    def normalize_rows(matrix):
        norms = np.linalg.norm(matrix, axis=1, keepdims=True)
        return np.divide(matrix, norms, out=np.zeros_like(matrix), where=norms > 0)

    fused = normalize_rows(normalize_rows(text_matrix) + normalize_rows(visual_matrix))
    similarity = fused @ fused.T
    neighbors: dict[int, dict[str, list[float]]] = {}
    for row_index, item_id in enumerate(item_order):
        row = similarity[row_index].copy()
        row[row_index] = -np.inf
        top_k = min(k, len(item_order) - 1)
        selected = np.argpartition(-row, top_k - 1)[:top_k]
        selected = sorted(selected.tolist(), key=lambda index: (-float(row[index]), item_order[index]))
        neighbors[item_id] = {
            "item_ids": [item_order[index] for index in selected],
            "scores": [float(row[index]) for index in selected],
        }
    return neighbors


# --------------------------------------------------------------------------
# NoDO: real transient LoRA perturbation hooks (hanorec/nodo/hooks.py port).
# --------------------------------------------------------------------------

def _iter_active_lora_pairs(model):
    for module_name, module in model.named_modules():
        lora_a = getattr(module, "lora_A", None)
        lora_b = getattr(module, "lora_B", None)
        if lora_a is None or lora_b is None:
            continue
        active = getattr(module, "active_adapters", None) or getattr(module, "active_adapter", None)
        if isinstance(active, str):
            active = [active]
        for adapter_name in active or []:
            if adapter_name in lora_a and adapter_name in lora_b:
                yield module_name, adapter_name, lora_a[adapter_name], lora_b[adapter_name]


class _NodoPerturbation:
    def __init__(self, model, sigma: float, torch_module, functional_module):
        if sigma < 0:
            raise ValueError("sigma must be non-negative")
        self._torch = torch_module
        self._functional = functional_module
        self._pairs = list(_iter_active_lora_pairs(model))
        if not self._pairs:
            raise RuntimeError("NoDO requires at least one active LoRA A/B module pair")
        self._handles: list[Any] = []

    def __enter__(self):
        torch_module = self._torch

        def make_hook(noise):
            def hook(_module, inputs, output):
                return output + self._functional.linear(inputs[0], noise, None)

            return hook

        for _module_name, _adapter_name, lora_a, lora_b in self._pairs:
            noise_a = torch_module.randn_like(lora_a.weight) * float(self._sigma)
            noise_b = torch_module.randn_like(lora_b.weight) * float(self._sigma)
            self._handles.append(lora_a.register_forward_hook(make_hook(noise_a)))
            self._handles.append(lora_b.register_forward_hook(make_hook(noise_b)))
        return self

    def __exit__(self, *exc_info):
        for handle in reversed(self._handles):
            handle.remove()
        self._handles = []
        return False


def perturb_lora_weights(model, sigma: float, torch_module, functional_module):
    perturbation = _NodoPerturbation(model, sigma, torch_module, functional_module)
    perturbation._sigma = sigma
    return perturbation


# --------------------------------------------------------------------------
# Main entry point.
# --------------------------------------------------------------------------

def train_and_evaluate(config: dict[str, Any], prepared: dict[str, Any], output_root: Path, hanorec_root: Path) -> dict[str, Any]:
    import torch
    import torch.nn.functional as functional
    from peft import LoraConfig, get_peft_model
    from transformers import AutoProcessor, BitsAndBytesConfig, Qwen2_5_VLForConditionalGeneration

    output_root.mkdir(parents=True, exist_ok=True)
    stage_timings: list[dict[str, Any]] = []

    def log_stage(name: str, seconds: float, units: int | None = None) -> None:
        payload: dict[str, Any] = {"stage": name, "seconds": seconds}
        if units:
            payload["units"] = units
            payload["seconds_per_unit"] = seconds / units
        stage_timings.append(payload)
        print(json.dumps({"timing": payload}))

    start_time = time.monotonic()
    deadline = start_time + float(config.get("budget_seconds", 6600))

    def remaining() -> float:
        return deadline - time.monotonic()

    catalog = prepared["catalog"]
    train_pairs = prepared["train"]
    evaluation_rows = prepared["evaluation"]
    max_pixels = int(config.get("max_pixels", 50176))
    top_k = min(10, len(catalog) - 1)

    model_load_start = time.monotonic()

    device_map = {"": 0}
    quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
    processor = AutoProcessor.from_pretrained(config["model_id"], revision=config["model_revision"])
    policy_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        config["model_id"],
        revision=config["model_revision"],
        quantization_config=quantization_config,
        device_map=device_map,
        torch_dtype=torch.float16,
    )
    log_stage("model_load", time.monotonic() - model_load_start)
    policy_model.eval()

    yes_token_id = processor.tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_token_id = processor.tokenizer.encode("No", add_special_tokens=False)[0]

    # -- Real semantic embeddings on the train-referenced subset only. --
    embedding_start = time.monotonic()
    item_order = sorted(int(item_id) for item_id in catalog.keys())
    text_vectors = []
    visual_vectors = []
    for item_id in item_order:
        record = catalog[str(item_id)]
        image = _resize_image(record["image_path"], max_pixels)
        text_vector, visual_vector = _semantic_embedding(policy_model, processor, torch, record["title"], image)
        text_vectors.append(text_vector)
        visual_vectors.append(visual_vector)
    log_stage("semantic_embedding", time.monotonic() - embedding_start, units=len(item_order))
    import numpy as np

    neighbors = _fuse_and_topk(np.stack(text_vectors), np.stack(visual_vectors), item_order, top_k)

    def pair_semantic_delta(positive_id: int, negative_id: int) -> float:
        positive_scores = neighbors[positive_id]["scores"]
        negative_scores = neighbors[negative_id]["scores"]
        return hars_probability_distance(hars_softmax(positive_scores), hars_softmax(negative_scores))

    semantic_deltas = [pair_semantic_delta(pair["positive"], pair["negative"]) for pair in train_pairs]
    lambda_sem_values = hars_normalize_hardness(semantic_deltas)
    cf_margins = [pair["cf_margin"] for pair in train_pairs]
    lambda_cf_values = cf_normalize_hardness(cf_margins)

    # -- SFT: LoRA adapter learns the yes/no format on the real positive framing. --
    lora_config = LoraConfig(
        r=8,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        task_type="CAUSAL_LM",
    )
    policy_model = get_peft_model(policy_model, lora_config)
    policy_model.train()
    optimizer = torch.optim.AdamW([p for p in policy_model.parameters() if p.requires_grad], lr=float(config["learning_rate"]))

    def build_inputs(history: list[int], candidate: int, shuffle_map: dict[int, int] | None):
        messages = _build_messages(catalog, history, candidate, max_pixels, shuffle_map)
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        images = [content["image"] for content in messages[0]["content"] if content["type"] == "image"]
        return processor(text=[text], images=images, return_tensors="pt").to(policy_model.device)

    def answer_logprobs(model, model_inputs) -> tuple:
        """One forward pass; returns (yes_logprob, no_logprob) predicted at the
        position right after the prompt. Reading both tokens from the same
        forward avoids a redundant second pass (previously one forward per
        token checked)."""
        outputs = model(**model_inputs)
        logits = outputs.logits[:, -1, :]
        log_probs = functional.log_softmax(logits.float(), dim=-1)
        return log_probs[0, yes_token_id], log_probs[0, no_token_id]

    def reference_answer_logprobs(model_inputs) -> tuple:
        """Real frozen-reference forward pass via PEFT's disable_adapter().

        get_peft_model() wraps the loaded model in place (shared tensors), so
        there is no independent second copy of the pretrained weights. The
        correct, standard way to get a deterministic pretrained-only forward
        pass from the same object is to disable the LoRA adapter and force
        eval mode (so dropout is inactive) for the duration of the call, then
        restore train mode so subsequent policy forwards behave as trained.
        """
        was_training = policy_model.training
        policy_model.eval()
        try:
            with torch.no_grad(), policy_model.disable_adapter():
                values = answer_logprobs(policy_model, model_inputs)
        finally:
            if was_training:
                policy_model.train()
        return values

    sft_start = time.monotonic()
    sft_steps = int(config.get("sft_steps", 2))
    sft_losses: list[float] = []
    for step in range(sft_steps):
        if remaining() <= 0:
            break
        optimizer.zero_grad()
        per_example_sft_losses: list[float] = []
        for pair in train_pairs:
            model_inputs = build_inputs(pair["history"], pair["positive"], None)
            yes_logprob, _no_logprob = answer_logprobs(policy_model, model_inputs)
            example_loss = -yes_logprob / len(train_pairs)
            example_loss.backward()
            per_example_sft_losses.append(float(example_loss.detach().cpu()))
        optimizer.step()
        sft_losses.append(sum(per_example_sft_losses))
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    log_stage("sft_total", time.monotonic() - sft_start, units=len(sft_losses) * len(train_pairs))

    sft_state = {name: value.detach().clone() for name, value in policy_model.named_parameters() if value.requires_grad}
    torch.save(sft_state, output_root / "sft_lora_state.pt")

    reference_probe_inputs = build_inputs(train_pairs[0]["history"], train_pairs[0]["positive"], None)
    reference_logprob_first = float(reference_answer_logprobs(reference_probe_inputs)[0].cpu())
    reference_logprob_second = float(reference_answer_logprobs(reference_probe_inputs)[0].cpu())
    if not math.isclose(reference_logprob_first, reference_logprob_second, abs_tol=1e-5):
        raise RuntimeError("reference model is not reproducible across calls; NoDO must never perturb it")

    def build_dpo_examples(shuffle_map: dict[int, int] | None):
        examples = []
        for pair in train_pairs:
            examples.append(
                {
                    "history": pair["history"],
                    "candidate": pair["positive"],
                    "chosen_token": yes_token_id,
                    "rejected_token": no_token_id,
                    "shuffle_map": shuffle_map,
                }
            )
            examples.append(
                {
                    "history": pair["history"],
                    "candidate": pair["negative"],
                    "chosen_token": no_token_id,
                    "rejected_token": yes_token_id,
                    "shuffle_map": shuffle_map,
                }
            )
        return examples

    rng = np.random.default_rng(int(config.get("seed", 2024)))
    permutation = item_order.copy()
    rng.shuffle(permutation)
    shuffle_map = dict(zip(item_order, permutation))

    dpo_steps = int(config.get("dpo_steps", 2))
    noise_sigma = float(config.get("noise_sigma", 0.05))
    beta0 = float(config.get("beta0", 0.1))
    arms_run: list[dict[str, Any]] = []

    for weight in config["weights"]:
        for image_condition, arm_shuffle_map in (("real", None), ("shuffle", shuffle_map)):
            if remaining() <= 0:
                arms_run.append({"weight": weight, "image_condition": image_condition, "status": "SKIPPED_BUDGET"})
                continue
            for name, value in policy_model.named_parameters():
                if value.requires_grad and name in sft_state:
                    value.data.copy_(sft_state[name])
            pre_training_snapshot = {name: value.detach().clone() for name, value in policy_model.named_parameters() if value.requires_grad}

            lambda_combined = [
                lambda_sem_values[i] ** weight * lambda_cf_values[i] ** (1.0 - weight) for i in range(len(train_pairs))
            ]
            if math.isclose(weight, 1.0):
                for computed, expected in zip(lambda_combined, lambda_sem_values):
                    if not math.isclose(computed, expected, rel_tol=1e-9):
                        raise RuntimeError("w=1.0 must reduce lambda_combined to lambda_sem exactly")
            if math.isclose(weight, 0.0):
                for computed, expected in zip(lambda_combined, lambda_cf_values):
                    if not math.isclose(computed, expected, rel_tol=1e-9):
                        raise RuntimeError("w=0.0 must reduce lambda_combined to lambda_cf exactly")

            examples = build_dpo_examples(arm_shuffle_map)
            hardness_per_example = [value for value in lambda_combined for _ in range(2)]

            step_losses: list[float] = []
            arm_stats_seconds = 0.0
            arm_train_seconds = 0.0
            for step in range(dpo_steps):
                if remaining() <= 0:
                    break
                # Pass 1: cheap no_grad batch statistics for Eq. 8 responsiveness/beta.
                # NoDO noise is intentionally omitted here (disclosed deviation) to keep
                # this pass memory-light; the training pass below applies NoDO for real.
                stats_pass_start = time.monotonic()
                stats_logits: list[float] = []
                for example in examples:
                    if remaining() <= 0:
                        break
                    chosen_inputs = build_inputs(example["history"], example["candidate"], example["shuffle_map"])
                    with torch.no_grad():
                        policy_yes, policy_no = answer_logprobs(policy_model, chosen_inputs)
                    reference_yes, reference_no = reference_answer_logprobs(chosen_inputs)
                    is_chosen_yes = example["chosen_token"] == yes_token_id
                    policy_chosen = policy_yes if is_chosen_yes else policy_no
                    policy_rejected = policy_no if is_chosen_yes else policy_yes
                    reference_chosen = reference_yes if is_chosen_yes else reference_no
                    reference_rejected = reference_no if is_chosen_yes else reference_yes
                    stats_logits.append(
                        float((policy_chosen - reference_chosen - policy_rejected + reference_rejected).cpu())
                    )
                arm_stats_seconds += time.monotonic() - stats_pass_start
                reward_gaps = [beta0 * value for value in stats_logits]
                if len(reward_gaps) < 3:
                    raise ValueError("HaNoRec Eq. 8 requires at least 3 examples in the global mini-batch")
                scale = responsiveness(reward_gaps)
                betas = [max(1e-6, beta0 * scale * hardness_per_example[i]) for i in range(len(examples))]

                # Pass 2: real per-example forward+backward, one example's graph alive at
                # a time, so activation memory never accumulates across the mini-batch.
                train_pass_start = time.monotonic()
                optimizer.zero_grad()
                per_example_losses: list[float] = []
                for index, example in enumerate(examples):
                    if remaining() <= 0:
                        break
                    chosen_inputs = build_inputs(example["history"], example["candidate"], example["shuffle_map"])
                    with perturb_lora_weights(policy_model, noise_sigma, torch, functional):
                        policy_yes, policy_no = answer_logprobs(policy_model, chosen_inputs)
                    reference_yes, reference_no = reference_answer_logprobs(chosen_inputs)
                    is_chosen_yes = example["chosen_token"] == yes_token_id
                    policy_chosen = policy_yes if is_chosen_yes else policy_no
                    policy_rejected = policy_no if is_chosen_yes else policy_yes
                    reference_chosen = reference_yes if is_chosen_yes else reference_no
                    reference_rejected = reference_no if is_chosen_yes else reference_yes
                    preference_logit = policy_chosen - reference_chosen - policy_rejected + reference_rejected
                    example_loss = -functional.logsigmoid(betas[index] * preference_logit) / len(examples)
                    if not torch.isfinite(example_loss):
                        raise RuntimeError("HaNoRec DPO loss produced a non-finite value")
                    example_loss.backward()
                    per_example_losses.append(float(example_loss.detach().cpu()))
                optimizer.step()
                step_losses.append(sum(per_example_losses))
                arm_train_seconds += time.monotonic() - train_pass_start
            log_stage(
                f"dpo_stats_pass_w{weight}_{image_condition}",
                arm_stats_seconds,
                units=len(step_losses) * len(examples),
            )
            log_stage(
                f"dpo_train_pass_w{weight}_{image_condition}",
                arm_train_seconds,
                units=len(step_losses) * len(examples),
            )
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            changed = any(
                not torch.equal(value, pre_training_snapshot[name])
                for name, value in policy_model.named_parameters()
                if value.requires_grad
            )
            if not changed and step_losses:
                raise RuntimeError("no LoRA parameter changed after DPO training; NoDO/backprop path is broken")

            arm_checkpoint_path = output_root / f"arm_w{weight}_{image_condition}.pt"
            arm_state = {name: value.detach().clone() for name, value in policy_model.named_parameters() if value.requires_grad}
            torch.save(arm_state, arm_checkpoint_path)

            predictions = []
            policy_model.eval()
            rerank_start = time.monotonic()
            total_candidates_scored = 0
            with torch.no_grad():
                for eval_row in evaluation_rows:
                    if remaining() <= 0:
                        break
                    candidate_scores = []
                    for candidate in eval_row["candidates"]:
                        candidate_inputs = build_inputs(eval_row["history"], candidate, arm_shuffle_map)
                        yes_logprob, no_logprob = answer_logprobs(policy_model, candidate_inputs)
                        candidate_scores.append(float((yes_logprob - no_logprob).cpu()))
                        total_candidates_scored += 1
                    ranked = sorted(zip(eval_row["candidates"], candidate_scores), key=lambda pair: -pair[1])
                    ranked_ids = [item for item, _ in ranked]
                    target = eval_row["target"]
                    rank = ranked_ids.index(target) + 1 if target in ranked_ids else 0
                    ndcg10 = (1.0 / math.log2(rank + 1)) if 0 < rank <= 10 else 0.0
                    recall10 = 1.0 if 0 < rank <= 10 else 0.0
                    candidate_recall20 = 1.0 if target in eval_row["candidates"] else 0.0
                    predictions.append(
                        {
                            "user_id": eval_row["user_id"],
                            "target": target,
                            "ranked_candidates": ranked_ids,
                            "rank": rank,
                            "ndcg@10": ndcg10,
                            "recall@10": recall10,
                            "candidate_recall@20": candidate_recall20,
                        }
                    )
            log_stage(f"rerank_w{weight}_{image_condition}", time.monotonic() - rerank_start, units=total_candidates_scored)
            policy_model.train()

            arms_run.append(
                {
                    "weight": weight,
                    "image_condition": image_condition,
                    "status": "COMPLETE" if step_losses else "SKIPPED_BUDGET",
                    "losses": step_losses,
                    "checkpoint": str(arm_checkpoint_path),
                    "predictions": predictions,
                    "mean_ndcg@10": sum(p["ndcg@10"] for p in predictions) / len(predictions) if predictions else None,
                    "mean_recall@10": sum(p["recall@10"] for p in predictions) / len(predictions) if predictions else None,
                    "mean_candidate_recall@20": (
                        sum(p["candidate_recall@20"] for p in predictions) / len(predictions) if predictions else None
                    ),
                }
            )

    (output_root / "hanorec_cf_hardness_result.json").write_text(
        json.dumps(
            {
                "sft_losses": sft_losses,
                "stage_timings": stage_timings,
                "lambda_sem": lambda_sem_values,
                "lambda_cf": lambda_cf_values,
                "arms": arms_run,
                "reference_reproducibility_check": {
                    "first": reference_logprob_first,
                    "second": reference_logprob_second,
                },
                "elapsed_seconds": time.monotonic() - start_time,
                "declared_deviations": [
                    "technical subset: HaRS Top-K computed only over the train-referenced catalog subset, "
                    "not the full Games catalog",
                    "tiny step counts (sft_steps/dpo_steps from config) for a 2 GPU-hour smoke budget, "
                    "not the paper's 5-epoch recipe",
                    "4-bit quantized Qwen2.5-VL-3B with float16 compute dtype, not the paper's full-precision setup",
                    "shuffle image condition is a global item-id permutation across the tiny train-referenced "
                    "subset, disclosed as technical-only, not frequency-matched",
                    "reranking uses per-candidate yes/no logit-margin scoring, not a pairwise tournament",
                    "Eq. 8 responsiveness/beta is computed from a no_grad statistics pass without NoDO noise "
                    "(kept memory-bounded on a single T4); the training pass applies NoDO independently per example",
                ],
            },
            indent=2,
        )
    )

    return {
        "sft_losses": sft_losses,
        "lambda_sem": lambda_sem_values,
        "lambda_cf": lambda_cf_values,
        "arms": arms_run,
        "reference_reproducibility_check": {"first": reference_logprob_first, "second": reference_logprob_second},
    }


In [ ]:
from pathlib import Path
Path("/kaggle/working/experiment.json").write_text('{\n  "seed": 2024,\n  "top_m": 20,\n  "train_pairs": 530,\n  "eval_users": 265,\n  "history_items": 3,\n  "sft_steps": 2,\n  "dpo_steps": 2,\n  "batch_size": 4,\n  "learning_rate": 0.0001,\n  "model_id": "Qwen/Qwen2.5-VL-3B-Instruct",\n  "model_revision": "66285546d2b821cf421d4f5eb2576359d3770cd3",\n  "weights": [\n    1.0,\n    0.0,\n    0.5\n  ],\n  "noise_sigma": 0.05,\n  "beta0": 0.1,\n  "max_pixels": 50176,\n  "budget_seconds": 41000,\n  "upstream": {\n    "llm2rec": {\n      "url": "https://codeload.github.com/HappyPointer/LLM2Rec/zip/73b481f710f67166ab958f4985d27b27fb410871",\n      "sha256": "6bcee7b09ec9ad5fa0107a083c7b8060abbead23e1264a49480906de638c1cc8",\n      "commit": "73b481f710f67166ab958f4985d27b27fb410871"\n    },\n    "hanorec": {\n      "url": "https://codeload.github.com/wangyu0627/HaNoRec/zip/587face74524e4553b5a7aa295fe962004682382",\n      "sha256": "48c52e5625cd4e25e80600fed9bed8aad8906556d267b87d52890b49bf33d6a6",\n      "commit": "587face74524e4553b5a7aa295fe962004682382"\n    }\n  },\n  "artifact_pins": {\n    "train_data.txt": "1300e5deec29d4bede6e32bfa1a5eade563f53a6916da95d67754ec40b9475b8",\n    "val_data.txt": "484a97cfbcff2a78e92612b15863ce08c84c285c7bfb325ec1f33d3884e6de16",\n    "test_data.txt": "80074ffea37928d92c35038e9d8bbf3de0cfcf01308ae4173e3c680c1bc50d1f",\n    "item_titles.json": "17f501809c4159905cfa6dfa5626d3d52d0be32bc9747ee89b003662f6f5e84c",\n    "LLM2Rec-budgeted-|kaggle|working|LLM2Rec|repeated_evaluate_with_seq-Sep-09-2026_11-12-13-c93f12.pth": "fb0d1d3916f10b5b6f0d9f597e7b2ac8437f445c15629ebf3099333b72806a59",\n    "Qwen2-0.5B-LLM2Rec-IEM-budgeted_step500_title_item_embs.npy": "89f9d3dd17f9ec49537f910e39c9761878299f7d24abdcf805ab0db91433c0f0",\n    "games_sasrec_step500.log": "a60181a26114072dd5442fd9ffe4b338505010d8b321d9e31ca1646b341ff154",\n    "Video_Games_image_manifest.jsonl": "5d9c586d2d05c6ff208c7a0f945a441a4fc60ef0d18ec8aacc269fbf67a42d04"\n  },\n  "checkpoint_name": "LLM2Rec-budgeted-|kaggle|working|LLM2Rec|repeated_evaluate_with_seq-Sep-09-2026_11-12-13-c93f12.pth",\n  "embedding_name": "Qwen2-0.5B-LLM2Rec-IEM-budgeted_step500_title_item_embs.npy",\n  "purpose": "calibration run for full-dataset GPU-hour sizing; not a research-effectiveness claim"\n}')


## Orchestration: real end-to-end run


In [ ]:
from pathlib import Path
import json as _json

EXPERIMENT_CONFIG = _json.loads(Path("/kaggle/working/experiment.json").read_text())
INPUT_ROOT = Path("/kaggle/input")
OUTPUT_ROOT = Path("/kaggle/working/hanorec_cf_hardness_output")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
EXPERIMENT_CONFIG["budget_seconds"] = max(60.0, DEADLINE - time.monotonic())
print({"effective_budget_seconds": EXPERIMENT_CONFIG["budget_seconds"]})

prepared = prepare(EXPERIMENT_CONFIG, INPUT_ROOT, OUTPUT_ROOT, Path("/kaggle/working"))
print({"train_pairs": len(prepared["train"]), "eval_rows": len(prepared["evaluation"]), "catalog_items": len(prepared["catalog"])})
(OUTPUT_ROOT / "prepared_manifest.json").write_text(_json.dumps({k: v for k, v in prepared.items() if k != "catalog"}, indent=2))

result = train_and_evaluate(EXPERIMENT_CONFIG, prepared, OUTPUT_ROOT, Path("/kaggle/working"))
print(_json.dumps(result, indent=2)[:4000])
